# Task 12: App vs. Website Time – Behavior Clustering & Conversion Prediction

**Objective:**
- Cluster user behavior patterns with KMeans and DBSCAN
- Detect multicollinearity using VIF and correlation analysis
- Build predictive models (Linear, Ridge, Lasso, Random Forest) to determine
  which time factor has a stronger impact on conversions / spending

**Input:** `data/processed/womens_clothing_scaled.csv`

## 1. Setup & Imports

In [ ]:
import os
import logging
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.inspection import permutation_importance
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

warnings.filterwarnings('ignore')

# ── Logging ────────────────────────────────────────────────────────────────────
os.makedirs('../../logs', exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler('../../logs/task_12.log', mode='w'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger('Task12')
logger.info('Task 12 started.')

# ── Dark theme ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a', 'axes.facecolor': '#1a1a2e',
    'axes.edgecolor': '#444466',   'axes.labelcolor': '#e0e0ff',
    'xtick.color': '#c0c0e0',      'ytick.color': '#c0c0e0',
    'text.color': '#e0e0ff',       'grid.color': '#2a2a4a',
    'grid.linestyle': '--',        'font.family': 'DejaVu Sans',
    'font.size': 11,               'axes.titlesize': 13,
    'axes.titleweight': 'bold',    'legend.facecolor': '#1a1a2e',
    'legend.edgecolor': '#444466',
})
PALETTE = ['#7c5cbf', '#e96479', '#f5a623', '#4fc3f7', '#81c784', '#ff8a65']
os.makedirs('../../reports', exist_ok=True)
print('✅ Setup complete.')

## 2. Load Data & Identify Columns

In [ ]:
df = pd.read_csv('../../data/processed/womens_clothing_scaled.csv')
logger.info(f'Dataset: {df.shape[0]} rows × {df.shape[1]} cols')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
# ── Identify app time, website time, and spending columns ─────────────────────
APP_COL    = None
WEB_COL    = None
SPEND_COL  = None
MEM_COL    = None   # membership length or session duration

for col in df.columns:
    cl = col.lower()
    if any(k in cl for k in ['app', 'mobile']) and APP_COL is None:
        if df[col].dtype in [np.float64, np.int64, float, int]:
            APP_COL = col
    if any(k in cl for k in ['web', 'site', 'website']) and WEB_COL is None:
        if df[col].dtype in [np.float64, np.int64, float, int]:
            WEB_COL = col
    if any(k in cl for k in ['spend', 'amount', 'purchase', 'total']) and SPEND_COL is None:
        if df[col].dtype in [np.float64, np.int64, float, int]:
            SPEND_COL = col
    if any(k in cl for k in ['member', 'length', 'session', 'duration']) and MEM_COL is None:
        if df[col].dtype in [np.float64, np.int64, float, int]:
            MEM_COL = col

# Fallback to numeric columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if APP_COL is None and len(num_cols) >= 1: APP_COL = num_cols[0]
if WEB_COL is None and len(num_cols) >= 2: WEB_COL = num_cols[1]
if SPEND_COL is None and len(num_cols) >= 3: SPEND_COL = num_cols[2]

print(f'App Time col    : {APP_COL}')
print(f'Website Time col: {WEB_COL}')
print(f'Spending col    : {SPEND_COL}')
print(f'Membership col  : {MEM_COL}')
logger.info(f'App:{APP_COL}, Web:{WEB_COL}, Spend:{SPEND_COL}, Mem:{MEM_COL}')

In [ ]:
# ── Prepare feature set ────────────────────────────────────────────────────────
feature_cols = [c for c in [APP_COL, WEB_COL, MEM_COL] if c]
all_cols = feature_cols + ([SPEND_COL] if SPEND_COL else [])
wdf = df[all_cols].dropna().copy()
print(f'Working rows: {len(wdf)}')

X = wdf[feature_cols].values
y = wdf[SPEND_COL].values if SPEND_COL else None

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 3. Raw Scatter – App Time vs. Website Time

In [ ]:
if APP_COL and WEB_COL:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(f'App Time vs. Website Time – Raw Exploration', fontsize=14, color='#e0e0ff')

    scatter_color = wdf[SPEND_COL].values if SPEND_COL else '#7c5cbf'
    sc = axes[0].scatter(wdf[APP_COL], wdf[WEB_COL],
                          c=scatter_color, cmap='plasma', alpha=0.4, s=15, edgecolors='none')
    if SPEND_COL:
        plt.colorbar(sc, ax=axes[0], label=SPEND_COL)
    axes[0].set_xlabel(APP_COL); axes[0].set_ylabel(WEB_COL)
    axes[0].set_title('Scatter colored by Spending')
    axes[0].yaxis.grid(True, alpha=0.3); axes[0].xaxis.grid(True, alpha=0.3)

    if SPEND_COL:
        # Spending tier color
        spend_q = wdf[SPEND_COL].quantile([0.33, 0.66])
        tier_colors = wdf[SPEND_COL].apply(
            lambda v: '#4fc3f7' if v <= spend_q[0.33] else ('#f5a623' if v <= spend_q[0.66] else '#e96479')
        )
        axes[1].scatter(wdf[APP_COL], wdf[WEB_COL],
                        c=tier_colors, alpha=0.4, s=15, edgecolors='none')
        patches = [
            mpatches.Patch(color='#4fc3f7', label='Low Spend'),
            mpatches.Patch(color='#f5a623', label='Mid Spend'),
            mpatches.Patch(color='#e96479', label='High Spend'),
        ]
        axes[1].legend(handles=patches)
    axes[1].set_xlabel(APP_COL); axes[1].set_ylabel(WEB_COL)
    axes[1].set_title('Scatter colored by Spending Tier')
    axes[1].yaxis.grid(True, alpha=0.3); axes[1].xaxis.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../../reports/task_12_raw_scatter.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()
    logger.info('Raw scatter saved.')

## 4. KMeans Clustering – User Behavior Patterns

In [ ]:
# ── Elbow + Silhouette ─────────────────────────────────────────────────────────
k_range   = range(2, 9)
inertias  = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels, sample_size=min(5000, len(X_scaled)))
    sil_scores.append(sil)
    print(f'  k={k}  inertia={km.inertia_:.1f}  silhouette={sil:.4f}')

best_k = list(k_range)[np.argmax(sil_scores)]
print(f'\nBest k (silhouette): {best_k}')
logger.info(f'Best k: {best_k}')

In [ ]:
# ── Elbow & Silhouette plots ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('KMeans – Model Selection', fontsize=14, color='#e0e0ff')

axes[0].plot(list(k_range), inertias, marker='o', color='#e96479', linewidth=2)
axes[0].axvline(best_k, color='#f5a623', linestyle='--', linewidth=1.5, label=f'Best k={best_k}')
axes[0].set_title('Elbow – Inertia'); axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia')
axes[0].legend(); axes[0].yaxis.grid(True, alpha=0.3)

axes[1].plot(list(k_range), sil_scores, marker='s', color='#4fc3f7', linewidth=2)
axes[1].axvline(best_k, color='#f5a623', linestyle='--', linewidth=1.5, label=f'Best k={best_k}')
axes[1].set_title('Silhouette Score'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette')
axes[1].legend(); axes[1].yaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../reports/task_12_kmeans_selection.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

In [ ]:
# ── Fit best KMeans ───────────────────────────────────────────────────────────
best_km = KMeans(n_clusters=best_k, random_state=42, n_init='auto')
wdf = wdf.copy()
wdf['KMeans_Cluster'] = best_km.fit_predict(X_scaled)

# ── Cluster profiles ──────────────────────────────────────────────────────────
agg_dict = {col: 'mean' for col in feature_cols}
if SPEND_COL: agg_dict[SPEND_COL] = 'mean'
agg_dict['KMeans_Cluster'] = 'count'

km_profiles = wdf.groupby('KMeans_Cluster').agg(
    Count=('KMeans_Cluster', 'count'),
    **{f'{col}_mean': (col, 'mean') for col in feature_cols + ([SPEND_COL] if SPEND_COL else [])}
).round(4).reset_index()

km_profiles.to_csv('../../reports/task_12_cluster_profiles.csv', index=False)
logger.info('KMeans cluster profiles saved.')
display(km_profiles)

In [ ]:
# ── Cluster scatter ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'KMeans Clusters – User Behavior Patterns (k={best_k})', fontsize=14, color='#e0e0ff')

cluster_colors = [PALETTE[i % len(PALETTE)] for i in wdf['KMeans_Cluster']]
axes[0].scatter(wdf[APP_COL], wdf[WEB_COL],
                c=cluster_colors, alpha=0.5, s=14, edgecolors='none')
axes[0].set_xlabel(APP_COL); axes[0].set_ylabel(WEB_COL)
axes[0].set_title(f'{APP_COL} vs {WEB_COL}')
patches = [mpatches.Patch(color=PALETTE[i % len(PALETTE)], label=f'Cluster {i}')
           for i in range(best_k)]
axes[0].legend(handles=patches); axes[0].yaxis.grid(True, alpha=0.3); axes[0].xaxis.grid(True, alpha=0.3)

# Cluster size
clus_counts = wdf['KMeans_Cluster'].value_counts().sort_index()
bar_colors = [PALETTE[i % len(PALETTE)] for i in clus_counts.index]
axes[1].bar([f'Cluster {i}' for i in clus_counts.index], clus_counts.values,
            color=bar_colors, edgecolor='white', linewidth=0.5)
for i, v in enumerate(clus_counts.values):
    axes[1].text(i, v + 5, f'{v:,}', ha='center', fontsize=9, color='#e0e0ff')
axes[1].set_title('Cluster Sizes'); axes[1].yaxis.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../../reports/task_12_cluster_scatter.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
logger.info('Cluster scatter saved.')

## 5. DBSCAN – Density-Based Clustering

In [ ]:
# ── DBSCAN ────────────────────────────────────────────────────────────────────
dbscan = DBSCAN(eps=0.5, min_samples=10)
db_labels = dbscan.fit_predict(X_scaled)

n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise       = (db_labels == -1).sum()
print(f'DBSCAN clusters: {n_clusters_db}  |  Noise points: {n_noise} ({n_noise/len(db_labels)*100:.1f}%)')
logger.info(f'DBSCAN: {n_clusters_db} clusters, {n_noise} noise points')

wdf['DBSCAN_Cluster'] = db_labels

# ── DBSCAN scatter ────────────────────────────────────────────────────────────
unique_labels = sorted(set(db_labels))
db_palette = {l: ('#888888' if l == -1 else PALETTE[i % len(PALETTE)])
              for i, l in enumerate(unique_labels)}

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0f0f1a')
for lbl in unique_labels:
    mask = db_labels == lbl
    ax.scatter(wdf[APP_COL][mask], wdf[WEB_COL][mask],
               c=db_palette[lbl], alpha=0.5 if lbl != -1 else 0.2,
               s=12 if lbl != -1 else 5, edgecolors='none',
               label='Noise' if lbl == -1 else f'Cluster {lbl}')
ax.set_xlabel(APP_COL); ax.set_ylabel(WEB_COL)
ax.set_title(f'DBSCAN Clustering – {n_clusters_db} clusters, {n_noise} noise', fontsize=13)
ax.legend(fontsize=8, loc='upper right')
ax.yaxis.grid(True, alpha=0.3); ax.xaxis.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../reports/task_12_dbscan_scatter.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
logger.info('DBSCAN scatter saved.')

## 6. Multicollinearity – VIF Analysis

In [ ]:
# ── Correlation heatmap ────────────────────────────────────────────────────────
corr = wdf[feature_cols].corr()

fig, ax = plt.subplots(figsize=(max(6, len(feature_cols)), max(5, len(feature_cols) * 0.9)))
fig.patch.set_facecolor('#0f0f1a')
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            ax=ax, linewidths=0.5, linecolor='#0f0f1a', cbar_kws={'label': 'Pearson r'})
ax.set_title('Feature Correlation Matrix – Time & Behavior Variables', fontsize=13)
plt.tight_layout()
plt.savefig('../../reports/task_12_correlation_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
logger.info('Correlation heatmap saved.')

In [ ]:
# ── Variance Inflation Factor (VIF) ───────────────────────────────────────────
X_vif = sm.add_constant(wdf[feature_cols])

vif_records = []
for i, col in enumerate(X_vif.columns):
    if col == 'const':
        continue
    vif_val = variance_inflation_factor(X_vif.values, i)
    severity = ('High – Remove/Transform' if vif_val > 10
                else 'Moderate – Monitor' if vif_val > 5
                else 'Acceptable')
    vif_records.append({'Feature': col, 'VIF': round(vif_val, 4), 'Severity': severity})
    print(f'{col:30s}: VIF={vif_val:.4f}  → {severity}')

vif_df = pd.DataFrame(vif_records)
vif_df.to_csv('../../reports/task_12_vif_analysis.csv', index=False)
logger.info('VIF analysis saved.')
display(vif_df)

In [ ]:
# ── VIF bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0f0f1a')
vif_colors = ['#e96479' if v > 10 else '#f5a623' if v > 5 else '#81c784'
              for v in vif_df['VIF']]
ax.barh(vif_df['Feature'], vif_df['VIF'], color=vif_colors,
        edgecolor='white', linewidth=0.5)
ax.axvline(5,  color='#f5a623', linestyle='--', linewidth=1.5, label='VIF=5 (Moderate threshold)')
ax.axvline(10, color='#e96479', linestyle='--', linewidth=1.5, label='VIF=10 (High threshold)')
for i, v in enumerate(vif_df['VIF']):
    ax.text(v + 0.1, i, f'{v:.2f}', va='center', fontsize=9, color='#e0e0ff')
ax.set_title('Variance Inflation Factor (VIF) – Multicollinearity Diagnostic', fontsize=13)
ax.set_xlabel('VIF'); ax.legend(fontsize=9)
ax.xaxis.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../../reports/task_12_vif_chart.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f1a')
plt.show()
logger.info('VIF chart saved.')

## 7. Predictive Models – Which Time Factor Matters Most?

In [ ]:
if y is None:
    print('⚠️  No spending/target column found – skipping predictive modeling.')
else:
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    models = {
        'Linear Regression': LinearRegression(),
        'Ridge (α=1)':       Ridge(alpha=1.0),
        'Ridge (α=10)':      Ridge(alpha=10.0),
        'Lasso (α=0.1)':     Lasso(alpha=0.1, max_iter=5000),
        'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
    }

    model_results = []
    fitted_models = {}

    for name, model in models.items():
        cv_r2   = cross_val_score(model, X_scaled, y, cv=kf, scoring='r2')
        cv_rmse = np.sqrt(-cross_val_score(model, X_scaled, y, cv=kf,
                                            scoring='neg_mean_squared_error'))
        model.fit(X_scaled, y)
        fitted_models[name] = model
        y_pred = model.predict(X_scaled)
        train_r2 = r2_score(y, y_pred)

        model_results.append({
            'Model': name,
            'CV_R2_Mean':   round(cv_r2.mean(), 4),
            'CV_R2_Std':    round(cv_r2.std(),  4),
            'CV_RMSE_Mean': round(cv_rmse.mean(), 4),
            'Train_R2':     round(train_r2, 4)
        })
        print(f'{name:25s}: CV R²={cv_r2.mean():.4f} ± {cv_r2.std():.4f}  RMSE={cv_rmse.mean():.4f}')

    model_df = pd.DataFrame(model_results)
    model_df.to_csv('../../reports/task_12_model_comparison.csv', index=False)
    logger.info('Model comparison saved.')
    display(model_df)

In [ ]:
if y is not None:
    # ── Model comparison bar chart ─────────────────────────────────────────────
    best_model_name = model_df.loc[model_df['CV_R2_Mean'].idxmax(), 'Model']
    bar_colors = ['#f5a623' if m == best_model_name else '#7c5cbf' for m in model_df['Model']]

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle('Model Comparison – Predicting Spending from Time Factors', fontsize=13, color='#e0e0ff')

    axes[0].barh(model_df['Model'], model_df['CV_R2_Mean'],
                 xerr=model_df['CV_R2_Std'], color=bar_colors,
                 edgecolor='white', linewidth=0.5, capsize=4)
    for i, (v, s) in enumerate(zip(model_df['CV_R2_Mean'], model_df['CV_R2_Std'])):
        axes[0].text(v + s + 0.001, i, f'{v:.4f}', va='center', fontsize=9, color='#e0e0ff')
    axes[0].set_title('Cross-Validated R²'); axes[0].set_xlabel('CV R²')
    axes[0].xaxis.grid(True, alpha=0.3)

    axes[1].barh(model_df['Model'], model_df['CV_RMSE_Mean'], color=bar_colors,
                 edgecolor='white', linewidth=0.5)
    for i, v in enumerate(model_df['CV_RMSE_Mean']):
        axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9, color='#e0e0ff')
    axes[1].set_title('Cross-Validated RMSE'); axes[1].set_xlabel('CV RMSE')
    axes[1].xaxis.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../../reports/task_12_model_comparison.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()

## 8. Feature Importance – Which Time Factor Wins?

In [ ]:
if y is not None:
    importance_records = []

    # ── Linear model coefficients ───────────────────────────────────────────────
    for name in ['Linear Regression', 'Ridge (α=1)', 'Lasso (α=0.1)']:
        model = fitted_models[name]
        for feat, coef in zip(feature_cols, model.coef_):
            importance_records.append({'Model': name, 'Feature': feat,
                                        'Importance': round(abs(coef), 6), 'Raw_Coef': round(coef, 6)})

    # ── Random Forest importances ───────────────────────────────────────────────
    rf = fitted_models['Random Forest']
    for feat, imp in zip(feature_cols, rf.feature_importances_):
        importance_records.append({'Model': 'Random Forest', 'Feature': feat,
                                    'Importance': round(imp, 6), 'Raw_Coef': round(imp, 6)})

    # ── Permutation importance (best model) ────────────────────────────────────
    best_fitted = fitted_models[best_model_name]
    perm = permutation_importance(best_fitted, X_scaled, y, n_repeats=10, random_state=42)
    for feat, imp_mean, imp_std in zip(feature_cols, perm.importances_mean, perm.importances_std):
        importance_records.append({'Model': f'Permutation ({best_model_name})', 'Feature': feat,
                                    'Importance': round(abs(imp_mean), 6), 'Raw_Coef': round(imp_mean, 6)})

    imp_df = pd.DataFrame(importance_records)
    imp_df.to_csv('../../reports/task_12_feature_importance.csv', index=False)
    logger.info('Feature importance saved.')

    # ── Feature importance grouped bar chart ────────────────────────────────────
    pivot_imp = imp_df.pivot_table(index='Feature', columns='Model', values='Importance', aggfunc='mean')
    # Normalize each column to [0,1] for fair comparison
    pivot_norm = pivot_imp.div(pivot_imp.sum(axis=0), axis=1)

    fig, ax = plt.subplots(figsize=(12, max(5, len(feature_cols) * 1.5)))
    fig.patch.set_facecolor('#0f0f1a')
    pivot_norm.plot(kind='barh', ax=ax,
                    color=PALETTE[:len(pivot_norm.columns)],
                    edgecolor='white', linewidth=0.4)
    ax.set_title('Feature Importance Comparison – All Models (Normalized)', fontsize=13)
    ax.set_xlabel('Normalized Importance'); ax.xaxis.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='lower right')
    plt.tight_layout()
    plt.savefig('../../reports/task_12_feature_importance.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()
    logger.info('Feature importance chart saved.')

    # ── Identify dominant time factor ──────────────────────────────────────────
    rf_imp = {feat: imp for feat, imp in zip(feature_cols, rf.feature_importances_)}
    app_imp = rf_imp.get(APP_COL, 0)
    web_imp = rf_imp.get(WEB_COL, 0)
    winner = APP_COL if app_imp > web_imp else WEB_COL
    print(f'\n🏆 Dominant Factor (Random Forest):')
    print(f'   {APP_COL}: {app_imp:.4f}')
    print(f'   {WEB_COL}: {web_imp:.4f}')
    print(f'   → {winner} has stronger impact on spending.')
    logger.info(f'Dominant factor: {winner}')

## 9. Partial Dependence – Manual Visualization

In [ ]:
if y is not None:
    rf = fitted_models['Random Forest']
    fig, axes = plt.subplots(1, len(feature_cols), figsize=(6 * len(feature_cols), 5))
    if len(feature_cols) == 1: axes = [axes]
    fig.suptitle('Partial Dependence Plots – Random Forest', fontsize=14, color='#e0e0ff')

    X_mean = X_scaled.mean(axis=0)

    for i, (ax, feat_col) in enumerate(zip(axes, feature_cols)):
        x_range = np.linspace(X_scaled[:, i].min(), X_scaled[:, i].max(), 100)
        X_temp = np.tile(X_mean, (100, 1))
        X_temp[:, i] = x_range
        y_pred_pd = rf.predict(X_temp)

        ax.plot(x_range, y_pred_pd, color=PALETTE[i % len(PALETTE)], linewidth=2.5)
        ax.fill_between(x_range, y_pred_pd, alpha=0.15, color=PALETTE[i % len(PALETTE)])
        ax.set_xlabel(feat_col + ' (scaled)'); ax.set_ylabel(f'Predicted {SPEND_COL}')
        ax.set_title(f'PDP: {feat_col}'); ax.yaxis.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('../../reports/task_12_partial_dependence.png', dpi=150,
                bbox_inches='tight', facecolor='#0f0f1a')
    plt.show()
    logger.info('Partial dependence plots saved.')

logger.info('Task 12 complete.')
print('\n✅ Task 12 complete. All reports saved to reports/ folder.')
print()
print('Output files:')
print('  📊 reports/task_12_cluster_profiles.csv')
print('  📊 reports/task_12_vif_analysis.csv')
print('  📊 reports/task_12_model_comparison.csv')
print('  📊 reports/task_12_feature_importance.csv')
print('  🖼️  reports/task_12_*.png  (8 plots)')